# Pizza shop data flow, as code

This notebook runs the zone-based data lake from `docs/data-flow.html` end to end on a
laptop. Nothing here needs a cloud account: the "lake" is a folder, the "warehouse" is a
DuckDB file, and the sources are small fake files that look like what the real POS and
delivery apps export.

```
Sources ──▶ LANDING ──▶ RAW ──▶ TRUSTED ──▶ REFINED ──▶ Consumers
            (as-is)    (Parquet, (star schema, (marts,     (dashboard,
                        forever)  cleaned)     aggregated) reorder list)
```

Each section is one zone. The pattern in every section is the same:
**what flows in, what we do to it, what we store, and why that zone exists.**

Run the cells top to bottom. Every run starts from an empty lake so the result is reproducible.

โน้ตบุ๊กนี้คือ data flow จากหน้า `docs/data-flow.th.html` ในรูปแบบโค้ดที่รันได้จริงบนเครื่องของเราเอง
ไม่ต้องมี cloud: "data lake" คือโฟลเดอร์หนึ่ง, "warehouse" คือไฟล์ DuckDB หนึ่งไฟล์
และข้อมูลต้นทางคือไฟล์ JSON ปลอมขนาดเล็กที่หน้าตาเหมือนที่ POS และแอปเดลิเวอรีส่งให้เราจริง ๆ

ทุก section คือหนึ่ง zone และเล่าเรื่องเดียวกันทุกครั้ง: **ข้อมูลอะไรไหลเข้ามา ทำอะไรกับมัน เก็บไว้แบบไหน และทำไม zone นี้ต้องมี**
รันเซลล์จากบนลงล่าง ทุกครั้งที่รันจะเริ่มจาก lake ว่าง ผลลัพธ์จึงเหมือนเดิมเสมอ

In [1]:
import json, shutil, textwrap
from pathlib import Path
from datetime import datetime, timedelta, timezone

import pandas as pd
import duckdb

BKK = "Asia/Bangkok"
LAKE = Path("lake")                      # the whole data lake lives in this folder  |  data lake ทั้งก้อนอยู่ในโฟลเดอร์นี้
ZONES = ["landing", "raw", "trusted", "refined"]

# start clean every run  |  เริ่มจาก lake ว่างทุกครั้งที่รัน
if LAKE.exists():
    shutil.rmtree(LAKE)
for z in ZONES:
    (LAKE / z).mkdir(parents=True)

def tree(root: Path):
    # print a folder as a tree so we can *see* each zone fill up  |  พิมพ์โฟลเดอร์เป็น tree เพื่อให้เห็นแต่ละ zone ค่อย ๆ มีข้อมูล
    for p in sorted(root.rglob("*")):
        depth = len(p.relative_to(root).parts) - 1
        mark = "/" if p.is_dir() else ""
        print("    " * depth + p.name + mark)

tree(LAKE)

landing/
raw/
refined/
trusted/


## 0 · Reference data (what the shop already knows)

Two small tables the transform step needs. In the real platform these live in Trusted as
`dim_menu_item` and `brg_recipe`; here they are typed in by hand so the notebook is self-contained.

- **Menu items** with a stable business key (`menu_item_nk`) and one row per size.
- **Recipes** in grams per unit sold. This is what turns "we sold 14 pepperoni pizzas" into
  "we used 2.1 kg of mozzarella".
- **Item name map**: each delivery app calls our pizza something different. Grab says
  "Large Pepperoni Pizza", LINE MAN says "Pepperoni Pizza (L)", the POS just says `PEP-L`.
  This map is the *only* place that knowledge lives.

ตารางอ้างอิงเล็ก ๆ สองชุดที่ขั้น transform ต้องใช้ ในระบบจริงจะอยู่ใน Trusted เป็น `dim_menu_item` และ `brg_recipe`
- **เมนู** มี key ถาวร (`menu_item_nk`) หนึ่งแถวต่อหนึ่งไซส์
- **สูตร** เป็นกรัมต่อหนึ่งชิ้นที่ขาย ตัวนี้แหละที่แปลง "ขายเปปเปอโรนีได้ 14 ถาด" เป็น "ใช้มอสซาเรลลาไป 2.1 กก."
- **ตารางแมปชื่อเมนู** แต่ละแอปเรียกพิซซ่าของเราไม่เหมือนกัน Grab เรียก "Large Pepperoni Pizza", LINE MAN เรียก "Pepperoni Pizza (L)", POS เรียก `PEP-L`
  ความรู้เรื่องนี้อยู่ที่ตารางนี้ **ที่เดียว** ไม่ต้องไปเขียนซ้ำในแดชบอร์ด

In [2]:
menu_items = pd.DataFrame([
    # menu_item_nk, item_name,       category, size, base_price_thb
    ("PEP-L", "Pepperoni",  "pizza", "L", 329),
    ("PEP-M", "Pepperoni",  "pizza", "M", 259),
    ("MAR-M", "Margherita", "pizza", "M", 219),
    ("HAW-L", "Hawaiian",   "pizza", "L", 309),
    ("COKE",  "Coke 325ml", "drink", "-",  35),
], columns=["menu_item_nk", "item_name", "category", "size_code", "base_price_thb"])

recipes = pd.DataFrame([
    # menu_item_nk, ingredient_nk, qty_per_unit (grams, or pieces for packaging)
    ("PEP-L", "flour_00",     280), ("PEP-L", "mozzarella", 150), ("PEP-L", "pepperoni", 60), ("PEP-L", "tomato_sauce", 90),
    ("PEP-M", "flour_00",     220), ("PEP-M", "mozzarella", 110), ("PEP-M", "pepperoni", 45), ("PEP-M", "tomato_sauce", 70),
    ("MAR-M", "flour_00",     220), ("MAR-M", "mozzarella", 110), ("MAR-M", "basil",      3), ("MAR-M", "tomato_sauce", 70),
    ("HAW-L", "flour_00",     280), ("HAW-L", "mozzarella", 150), ("HAW-L", "ham",       60), ("HAW-L", "pineapple",   80), ("HAW-L", "tomato_sauce", 90),
    ("COKE",  "coke_can",       1),
], columns=["menu_item_nk", "ingredient_nk", "qty_per_unit"])

# how each channel names our items -> our key.  POS already uses our keys, so it is not here.
# แต่ละช่องทางเรียกเมนูเราว่าอะไร -> key ของเรา  POS ใช้ key เราอยู่แล้วเลยไม่ต้องอยู่ในนี้
item_name_map = pd.DataFrame([
    ("GRAB",    "Large Pepperoni Pizza",    "PEP-L"),
    ("GRAB",    "Medium Pepperoni Pizza",   "PEP-M"),
    ("GRAB",    "Medium Margherita Pizza",  "MAR-M"),
    ("GRAB",    "Large Hawaiian Pizza",     "HAW-L"),
    ("GRAB",    "Coca-Cola 325ml",          "COKE"),
    ("LINEMAN", "Pepperoni Pizza (L)",      "PEP-L"),
    ("LINEMAN", "Pepperoni Pizza (M)",      "PEP-M"),
    ("LINEMAN", "Margherita Pizza (M)",     "MAR-M"),
    ("LINEMAN", "Hawaiian Pizza (L)",       "HAW-L"),
    ("LINEMAN", "Coke",                     "COKE"),
], columns=["channel_code", "platform_item_name", "menu_item_nk"])

menu_items

,menu_item_nk,item_name,category,size_code,base_price_thb
0,PEP-L,Pepperoni,pizza,L,329
1,PEP-M,Pepperoni,pizza,M,259
2,MAR-M,Margherita,pizza,M,219
3,HAW-L,Hawaiian,pizza,L,309
4,COKE,Coke 325ml,drink,-,35


## 1 · Sources (the operational systems)

These files are **not** part of the lake. They are what the POS and the delivery apps hand us.
We fake them here, with the same quirks as the real thing:

| Source | Format | Timestamps | Quirk we must handle |
|---|---|---|---|
| POS / QR | one JSON file every 5 minutes | ISO with `+07:00` | an order appears **twice** (once when placed, again when paid) |
| Grab | one JSON export next morning | **UTC**, `Z` suffix | one order object is repeated; one order is cancelled |
| LINE MAN | one JSON export next morning | local time, `dd/mm/yyyy HH:MM` | different field names for everything, nested differently |

All three are JSON, but that does not make them the same shape. Every field name, timestamp format
and nesting is different, which is exactly why the conform step in Trusted exists.

The business day being demonstrated is **Friday 2026-09-11**. The shop trades 10:00 to 02:00,
so an order at 00:40 on Saturday still belongs to Friday. Watch for order `P-1012`.

> The customer never waits for any of this. The POS already printed the receipt. These files are
> copies of what already happened.

ไฟล์พวกนี้ **ไม่ใช่** ส่วนหนึ่งของ lake แต่เป็นสิ่งที่ POS และแอปเดลิเวอรีส่งมาให้ เราจำลองขึ้นมาพร้อมข้อผิดพลาดแบบเดียวกับของจริง:
- **POS / QR**: ไฟล์ JSON ทุก 5 นาที เวลาเป็น `+07:00` ออร์เดอร์เดียวกัน **โผล่สองครั้ง** (ตอนสั่ง และตอนจ่ายเงิน)
- **Grab**: JSON ส่งออกเช้าวันถัดไป เวลาเป็น **UTC** (ลงท้าย `Z`) มีออร์เดอร์หนึ่งรายการโผล่ซ้ำ และมีออร์เดอร์ที่ถูกยกเลิก
- **LINE MAN**: JSON เวลาเป็น local แบบ `dd/mm/yyyy HH:MM` ชื่อ field และการซ้อนโครงสร้างต่างจาก Grab หมด

ทั้งสามเป็น JSON เหมือนกัน แต่ไม่ได้แปลว่าหน้าตาเหมือนกัน ชื่อ field รูปแบบเวลา และการซ้อนต่างกันหมด นี่คือเหตุผลที่ต้องมีขั้น conform ใน Trusted

วันทำการที่ใช้สาธิตคือ **ศุกร์ 2026-09-11** ร้านเปิด 10:00–02:00 ดังนั้นออร์เดอร์ตอน 00:40 ของวันเสาร์ยังนับเป็นวันศุกร์ ให้สังเกตออร์เดอร์ `P-1012`

> ลูกค้าไม่ต้องรออะไรทั้งนั้น POS พิมพ์ใบเสร็จไปแล้ว ไฟล์เหล่านี้คือ *สำเนา* ของสิ่งที่เกิดขึ้นไปแล้ว

In [3]:
SRC = Path("sources_out")               # pretend this is the POS server / merchant portal  |  สมมติว่านี่คือเซิร์ฟเวอร์ POS และ merchant portal
if SRC.exists():
    shutil.rmtree(SRC)
(SRC / "pos").mkdir(parents=True)
(SRC / "grab").mkdir()
(SRC / "lineman").mkdir()

def bkk(y, m, d, hh, mm):
    return pd.Timestamp(year=y, month=m, day=d, hour=hh, minute=mm, tz=BKK)

# ---- POS: orders through the day, emitted into 5-minute micro-batch files -----------------
pos_orders = [
    # id,       time,                    channel,    lines [(sku, qty)]
    ("P-1001", bkk(2026,9,11,11,12), "QR_DINEIN", [("MAR-M",1),("COKE",2)]),
    ("P-1002", bkk(2026,9,11,12,5), "PHONE",     [("PEP-L",2)]),
    ("P-1003", bkk(2026,9,11,12,48), "QR_DINEIN", [("HAW-L",1),("COKE",1)]),
    ("P-1004", bkk(2026,9,11,18,31), "QR_DINEIN", [("PEP-M",1),("MAR-M",1)]),
    ("P-1005", bkk(2026,9,11,19,2), "QR_DINEIN", [("PEP-L",1),("COKE",3)]),
    ("P-1006", bkk(2026,9,11,19,40), "PHONE",     [("PEP-L",1),("HAW-L",1)]),
    ("P-1007", bkk(2026,9,11,20,15), "QR_DINEIN", [("PEP-M",2)]),
    ("P-1008", bkk(2026,9,11,21,3), "QR_DINEIN", [("MAR-M",1)]),
    ("P-1009", bkk(2026,9,11,22,50), "QR_DINEIN", [("PEP-L",1),("COKE",1)]),
    ("P-1012", bkk(2026,9,12, 0,40), "QR_DINEIN", [("PEP-M",1),("COKE",2)]),   # after midnight: still Friday  |  หลังเที่ยงคืน แต่ยังเป็นวันศุกร์
]
price = dict(zip(menu_items.menu_item_nk, menu_items.base_price_thb))

def pos_record(oid, ts, ch, lines, status):
    return {
        "pos_order_id": oid,
        "ordered_at": ts.isoformat(),
        "channel": ch,
        "status": status,
        "lines": [{"sku": s, "qty": q, "unit_price": price[s]} for s, q in lines],
        "total": sum(price[s]*q for s, q in lines),
    }

# the POS exports every 5 min. an order shows up when PLACED and again a few minutes later when PAID.
# POS ส่งออกทุก 5 นาที ออร์เดอร์โผล่ตอน PLACED และโผล่อีกครั้งตอน PAID (นี่คือที่มาของแถวซ้ำ)
events = []
for oid, ts, ch, lines in pos_orders:
    events.append((ts,                          pos_record(oid, ts, ch, lines, "PLACED")))
    events.append((ts + timedelta(minutes=3),   pos_record(oid, ts, ch, lines, "PAID")))
batches = {}
for ts, rec in events:
    slot = ts.floor("5min")
    batches.setdefault(slot, []).append(rec)
for slot, recs in batches.items():
    fname = SRC / "pos" / f"orders_{slot.strftime('%Y%m%dT%H%M')}.json"
    fname.write_text(json.dumps(recs, indent=1))

# ---- Grab: merchant-portal JSON export, Saturday 06:00, one object per order, times in UTC ----
grab_price = {"Large Pepperoni Pizza": 359, "Medium Pepperoni Pizza": 285, "Medium Margherita Pizza": 239,
              "Large Hawaiian Pizza": 339, "Coca-Cola 325ml": 40}   # platform prices are uplifted  |  ราคาบนแอปบวกเพิ่มจากหน้าร้าน
def grab_order(oid, ts, status, items):
    return {"orderID": oid,
            "orderTime": ts.tz_convert("UTC").strftime("%Y-%m-%dT%H:%M:%SZ"),
            "orderStatus": status,
            "items": [{"itemName": n, "quantity": q, "itemPrice": grab_price[n]} for n, q in items],
            "commissionPct": 30}
grab_orders = [
    grab_order("GF-88101", bkk(2026,9,11,18,5),  "COMPLETED", [("Medium Margherita Pizza", 1), ("Coca-Cola 325ml", 2)]),
    grab_order("GF-88117", bkk(2026,9,11,19,42), "COMPLETED", [("Large Pepperoni Pizza", 1)]),   # the order traced in the HTML page  |  ออร์เดอร์ที่ไล่ตามในหน้า HTML
    grab_order("GF-88117", bkk(2026,9,11,19,42), "COMPLETED", [("Large Pepperoni Pizza", 1)]),   # <- same order exported twice by the portal  |  ออร์เดอร์เดียวกัน portal ส่งออกซ้ำสองครั้ง
    grab_order("GF-88130", bkk(2026,9,11,20,27), "CANCELLED", [("Large Hawaiian Pizza", 1)]),
    grab_order("GF-88144", bkk(2026,9,11,21,10), "COMPLETED", [("Medium Pepperoni Pizza", 2)]),
    grab_order("GF-88151", bkk(2026,9,12,0,15),  "COMPLETED", [("Large Pepperoni Pizza", 1)]),
]
(SRC / "grab" / "orders_2026-09-11.json").write_text(json.dumps(grab_orders, indent=1))

# ---- LINE MAN: different portal, different everything  |  LINE MAN: คนละ portal คนละชื่อ field --------------------------------------
lm_orders = [
    {"order_no": "LM-5501", "created_at": "11/09/2026 12:20", "menu": [{"name": "Pepperoni Pizza (L)", "qty": 1, "subtotal": 359},
                                                                       {"name": "Coke",                "qty": 1, "subtotal": 40}]},
    {"order_no": "LM-5533", "created_at": "11/09/2026 19:55", "menu": [{"name": "Hawaiian Pizza (L)",   "qty": 1, "subtotal": 339}]},
    {"order_no": "LM-5540", "created_at": "11/09/2026 20:41", "menu": [{"name": "Margherita Pizza (M)", "qty": 2, "subtotal": 478}]},
    {"order_no": "LM-5562", "created_at": "11/09/2026 23:30", "menu": [{"name": "Pepperoni Pizza (M)",  "qty": 1, "subtotal": 285}]},
]
(SRC / "lineman" / "export_20260912.json").write_text(json.dumps({"exported_at": "12/09/2026 06:00", "orders": lm_orders}, indent=1))

print("What the source systems hand us:\n")
tree(SRC)
print("\nOne Grab order looks like this (note UTC time and Grab's item name):")
print(json.dumps(grab_orders[1], indent=2))

What the source systems hand us:

grab/
    orders_2026-09-11.json
lineman/
    export_20260912.json
pos/
    orders_20260911T1110.json
    orders_20260911T1115.json
    orders_20260911T1205.json
    orders_20260911T1245.json
    orders_20260911T1250.json
    orders_20260911T1830.json
    orders_20260911T1900.json
    orders_20260911T1905.json
    orders_20260911T1940.json
    orders_20260911T2015.json
    orders_20260911T2100.json
    orders_20260911T2105.json
    orders_20260911T2250.json
    orders_20260912T0040.json

One Grab order looks like this (note UTC time and Grab's item name):
{
  "orderID": "GF-88117",
  "orderTime": "2026-09-11T12:42:00Z",
  "orderStatus": "COMPLETED",
  "items": [
    {
      "itemName": "Large Pepperoni Pizza",
      "quantity": 1,
      "itemPrice": 359
    }
  ],
  "commissionPct": 30
}


## 2 · Landing zone

**Flows in:** files exactly as the sources produce them.
**Processed:** nothing. We only check the file arrived and has a size.
**Stored:** native JSON / CSV, one folder per source per day. Deleted after 7 days.

**Why it exists:** it is a safe drop point. Grab's export does not need to know anything about our
tables. If Grab renames a column, the file still lands, and we fix the adapter in the next zone
without losing the file.

Cadence differs by source, and that is the whole point of the "real time or batch" discussion:
POS files arrive every 5 minutes during service, delivery exports arrive once next morning.

**ไหลเข้า:** ไฟล์ตามที่ต้นทางส่งมาเป๊ะ ๆ **ประมวลผล:** ไม่ทำอะไร แค่เช็กว่าไฟล์มาถึงและไม่ว่าง
**เก็บเป็น:** JSON ดั้งเดิม แยกโฟลเดอร์ตามแหล่งและวัน ลบทิ้งหลัง 7 วัน

**ทำไมต้องมี:** เป็นจุดวางไฟล์ที่ปลอดภัย Grab ไม่จำเป็นต้องรู้โครงสร้างตารางของเรา ถ้า Grab เปลี่ยนชื่อคอลัมน์ ไฟล์ก็ยังลงจอดได้
แล้วเราค่อยแก้ adapter ใน zone ถัดไปโดยไม่เสียไฟล์

ความถี่ต่างกันตามแหล่ง และนี่คือหัวใจของคำถาม "real time หรือ batch": POS มาทุก 5 นาทีระหว่างเปิดร้าน ส่วนแอปเดลิเวอรีมาครั้งเดียวเช้าวันถัดไป

In [4]:
def land(source: str, file: Path, arrived_at: pd.Timestamp) -> Path:
    # copy as-is into landing/<source>/<arrival date>/<same file name>
    # copy ตามเดิมเป๊ะ ๆ ไปที่ landing/<แหล่ง>/<วันที่มาถึง>/<ชื่อไฟล์เดิม>
    dest = LAKE / "landing" / source / arrived_at.strftime("%Y-%m-%d") / file.name
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(file, dest)
    assert dest.stat().st_size > 0, f"empty file landed: {dest}"
    return dest

# POS: each micro-batch lands 5 minutes after its slot, on the day it was produced
# POS: แต่ละ micro-batch มาถึง 5 นาทีหลังช่วงเวลาของมัน ในวันเดียวกัน
for f in sorted((SRC / "pos").glob("*.json")):
    slot = pd.Timestamp(f.stem.split("_")[1], tz=BKK)
    land("pos", f, slot + timedelta(minutes=5))

# delivery apps: portal exports land the next morning (T+1)  |  แอปเดลิเวอรี: export มาถึงเช้าวันถัดไป
land("grab",    SRC / "grab" / "orders_2026-09-11.json",  bkk(2026,9,12,6,0))
land("lineman", SRC / "lineman" / "export_20260912.json", bkk(2026,9,12,6,0))

tree(LAKE / "landing")

grab/
    2026-09-12/
        orders_2026-09-11.json
lineman/
    2026-09-12/
        export_20260912.json
pos/
    2026-09-11/
        orders_20260911T1110.json
        orders_20260911T1115.json
        orders_20260911T1205.json
        orders_20260911T1245.json
        orders_20260911T1250.json
        orders_20260911T1830.json
        orders_20260911T1900.json
        orders_20260911T1905.json
        orders_20260911T1940.json
        orders_20260911T2015.json
        orders_20260911T2100.json
        orders_20260911T2105.json
        orders_20260911T2250.json
    2026-09-12/
        orders_20260912T0040.json


## 3 · Raw zone

**Flows in:** a copy of every landed file.
**Processed:** tag each row with `source_system`, `ingest_ts`, `source_file`. Nothing else.
No typing, no dedup, no joins.
**Stored:** Parquet with one `payload` column holding the original JSON, partitioned by source and date. Kept forever.

**Why it exists:** this is the audit trail and the replay button. Notice below that the
repeated Grab order and the twice-emitted POS orders are **still there**. That is correct.
Raw records what arrived, not what we think happened. If the cleaning logic has a bug next month,
we fix the bug and re-run from here; we never have to ask Grab for the file again.

**ไหลเข้า:** สำเนาของทุกไฟล์ใน Landing **ประมวลผล:** แปะ `source_system`, `ingest_ts`, `source_file` ให้ทุกแถว แค่นั้น
ไม่แปลง type ไม่ตัดแถวซ้ำ ไม่ join **เก็บเป็น:** Parquet ที่มีคอลัมน์ `payload` เก็บ JSON ต้นฉบับ แบ่ง partition ตามแหล่งและวัน เก็บตลอดไป

**ทำไมต้องมี:** นี่คือหลักฐานตรวจสอบย้อนหลังและ "ปุ่มรันใหม่" สังเกตว่าออร์เดอร์ Grab ที่โผล่ซ้ำ และออร์เดอร์ POS ที่โผล่สองครั้ง **ยังอยู่ครบ** ซึ่งถูกต้องแล้ว
Raw บันทึกสิ่งที่ *มาถึง* ไม่ใช่สิ่งที่เราคิดว่าเกิดขึ้น ถ้าเดือนหน้าเจอบั๊กใน logic ทำความสะอาด เราแก้บั๊กแล้วรันใหม่จากตรงนี้ ไม่ต้องขอไฟล์จาก Grab อีก

In [5]:
def read_landed(path: Path) -> pd.DataFrame:
    # the only source-specific thing in Raw is *where the list of orders sits in the file*
    # สิ่งเดียวที่ Raw รู้เกี่ยวกับแต่ละแหล่งคือ *list ของออร์เดอร์อยู่ตรงไหนในไฟล์*
    doc = json.loads(path.read_text())
    recs = doc["orders"] if isinstance(doc, dict) else doc       # LINE MAN wraps its list, the others do not  |  LINE MAN ห่อ list ไว้อีกชั้น เจ้าอื่นไม่ห่อ
    return pd.DataFrame({"payload": [json.dumps(r) for r in recs]})    # keep the whole record as text, nothing cast  |  เก็บทั้งเรคคอร์ดเป็น text ยังไม่แปลง type

def ingest(landed: Path, source: str, ingest_ts: pd.Timestamp) -> Path:
    df = read_landed(landed)
    df.insert(0, "source_system", source.upper())
    df.insert(1, "ingest_ts", ingest_ts)
    df.insert(2, "source_file", landed.name)
    out = LAKE / "raw" / source / f"ingest_date={ingest_ts.strftime('%Y-%m-%d')}" / (landed.stem + ".parquet")
    out.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(out, index=False)
    return out

for landed in sorted((LAKE / "landing").rglob("*.*")):
    source = landed.relative_to(LAKE / "landing").parts[0]
    arrival_day = landed.relative_to(LAKE / "landing").parts[1]
    # POS is ingested right away; delivery exports at 06:05 the morning they land
    # POS นำเข้าทันที ส่วน export ของแอปนำเข้าตอน 06:05 เช้าที่ไฟล์มาถึง
    ingest_ts = pd.Timestamp(arrival_day, tz=BKK) + (timedelta(hours=6, minutes=5) if source != "pos" else timedelta(hours=12))
    ingest(landed, source, ingest_ts)

tree(LAKE / "raw")

grab/
    ingest_date=2026-09-12/
        orders_2026-09-11.parquet
lineman/
    ingest_date=2026-09-12/
        export_20260912.parquet
pos/
    ingest_date=2026-09-11/
        orders_20260911T1110.parquet
        orders_20260911T1115.parquet
        orders_20260911T1205.parquet
        orders_20260911T1245.parquet
        orders_20260911T1250.parquet
        orders_20260911T1830.parquet
        orders_20260911T1900.parquet
        orders_20260911T1905.parquet
        orders_20260911T1940.parquet
        orders_20260911T2015.parquet
        orders_20260911T2100.parquet
        orders_20260911T2105.parquet
        orders_20260911T2250.parquet
    ingest_date=2026-09-12/
        orders_20260912T0040.parquet


In [6]:
con = duckdb.connect(str(LAKE / "trusted" / "warehouse.duckdb"))
for schema in ["trusted", "refined"]:
    con.execute(f"CREATE SCHEMA IF NOT EXISTS {schema}")

print("Raw Grab, exactly as it arrived (still repeated, still UTC, still one JSON string per order):")
display(con.execute("""
    SELECT source_file,
           json_extract_string(payload, '$.orderID')   AS order_id,
           json_extract_string(payload, '$.orderTime') AS order_time_utc,
           json_extract_string(payload, '$.items[0].itemName') AS first_item,
           payload
    FROM read_parquet('lake/raw/grab/*/*.parquet')
""").df())

print("Raw POS row count vs real orders:")
display(con.execute("""
    SELECT count(*) AS raw_rows,
           count(DISTINCT json_extract_string(payload, '$.pos_order_id')) AS distinct_orders
    FROM read_parquet('lake/raw/pos/*/*.parquet')
""").df())

Raw Grab, exactly as it arrived (still repeated, still UTC, still one JSON string per order):


,source_file,order_id,order_time_utc,first_item,payload
0,orders_2026-09-11.json,GF-88101,2026-09-11T11:05:00Z,Medium Margherita Pizza,"{""orderID"": ""GF-88101"", ""orderTime"": ""2026-09-..."
1,orders_2026-09-11.json,GF-88117,2026-09-11T12:42:00Z,Large Pepperoni Pizza,"{""orderID"": ""GF-88117"", ""orderTime"": ""2026-09-..."
2,orders_2026-09-11.json,GF-88117,2026-09-11T12:42:00Z,Large Pepperoni Pizza,"{""orderID"": ""GF-88117"", ""orderTime"": ""2026-09-..."
3,orders_2026-09-11.json,GF-88130,2026-09-11T13:27:00Z,Large Hawaiian Pizza,"{""orderID"": ""GF-88130"", ""orderTime"": ""2026-09-..."
4,orders_2026-09-11.json,GF-88144,2026-09-11T14:10:00Z,Medium Pepperoni Pizza,"{""orderID"": ""GF-88144"", ""orderTime"": ""2026-09-..."
5,orders_2026-09-11.json,GF-88151,2026-09-11T17:15:00Z,Large Pepperoni Pizza,"{""orderID"": ""GF-88151"", ""orderTime"": ""2026-09-..."


Raw POS row count vs real orders:


,raw_rows,distinct_orders
0,20,10


## 4 · Trusted zone

This is the nightly 03:00 job. It runs once for one **business day** and does everything the
README's staging and core layers promise:

1. **Parse and cast.** Text becomes numbers and timestamps.
2. **One timezone.** Grab's UTC and LINE MAN's local strings both become `Asia/Bangkok`.
3. **Business day.** Anything before 10:00 belongs to the previous day.
4. **Conform names.** "Large Pepperoni Pizza" and "Pepperoni Pizza (L)" both become `PEP-L`.
5. **Dedupe on the natural key.** POS: latest snapshot per `pos_order_id`.
   Grab: `(platform, platform_order_id)` plus the repeated order object dropped.
6. **Quality gates.** Every line has a channel, a business date, a known item, a positive quantity.
   If a gate fails the job stops and nothing is published.
7. **Load a star schema**: `dim_channel`, `dim_menu_item`, `fct_order`, `fct_order_line`.

**Why it exists:** every business rule lives here **once**. The dashboard, the reorder list and
the forecast model never reimplement "what is a business day" or "which name means which pizza".

นี่คืองาน batch ตอนตี 3 รันครั้งละหนึ่ง **วันทำการ** และทำทุกอย่างที่ชั้น staging + core ใน README สัญญาไว้:
1. **Parse และ cast** ข้อความกลายเป็นตัวเลขและ timestamp
2. **timezone เดียว** UTC ของ Grab และ string ของ LINE MAN กลายเป็น `Asia/Bangkok` ทั้งคู่
3. **วันทำการ** อะไรก่อน 10:00 นับเป็นวันก่อนหน้า
4. **ชื่อเมนูมาตรฐานเดียว** "Large Pepperoni Pizza" และ "Pepperoni Pizza (L)" กลายเป็น `PEP-L` ทั้งคู่
5. **ตัดแถวซ้ำด้วย natural key** POS: เอา snapshot ล่าสุดต่อ `pos_order_id`; Grab: `(platform, platform_order_id)` และตัด object ออร์เดอร์ที่ซ้ำ
6. **ด่านตรวจคุณภาพ** ทุกบรรทัดต้องมี channel, มีวันทำการ, เมนูต้องรู้จัก, จำนวนต้องเป็นบวก ถ้าไม่ผ่าน งานหยุดและไม่เผยแพร่อะไรเลย
7. **โหลดเข้า star schema**: `dim_channel`, `dim_menu_item`, `fct_order`, `fct_order_line`

**ทำไมต้องมี:** กฎธุรกิจทุกข้ออยู่ที่นี่ **ครั้งเดียว** แดชบอร์ด รายการสั่งของ และโมเดลพยากรณ์ไม่ต้องมานิยาม "วันทำการคืออะไร" หรือ "ชื่อไหนคือพิซซ่าอะไร" เองอีก

In [7]:
def business_date(ts: pd.Series) -> pd.Series:
    # the shop trades 10:00-02:00; anything before 10:00 belongs to the previous day
    # ร้านเปิด 10:00-02:00 อะไรก่อน 10:00 นับเป็นวันก่อนหน้า
    return (ts - pd.Timedelta(hours=10)).dt.date

# ---- one adapter per source: raw columns -> one common shape  |  หนึ่ง adapter ต่อหนึ่งแหล่ง: คอลัมน์ดิบ -> รูปแบบกลางเดียว --------------------------------
COMMON = ["source_system", "external_order_id", "channel_code", "order_ts", "order_status",
          "platform_item_name", "menu_item_nk", "quantity", "unit_price_thb", "ingest_ts", "source_file"]

def adapt_pos(raw: pd.DataFrame) -> pd.DataFrame:
    recs = raw.payload.map(json.loads)
    rows = []
    for meta, r in zip(raw.itertuples(), recs):
        for ln in r["lines"]:
            rows.append(dict(source_system="POS", external_order_id=r["pos_order_id"], channel_code=r["channel"],
                             order_ts=pd.Timestamp(r["ordered_at"]).tz_convert(BKK), order_status=r["status"],
                             platform_item_name=ln["sku"], menu_item_nk=ln["sku"],
                             quantity=int(ln["qty"]), unit_price_thb=float(ln["unit_price"]),
                             ingest_ts=meta.ingest_ts, source_file=meta.source_file))
    return pd.DataFrame(rows, columns=COMMON)

def adapt_grab(raw: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for meta, r in zip(raw.itertuples(), raw.payload.map(json.loads)):
        for it in r["items"]:
            rows.append(dict(source_system="GRAB", external_order_id=r["orderID"], channel_code="GRAB",
                             order_ts=pd.Timestamp(r["orderTime"]).tz_convert(BKK),         # 'Z' suffix -> UTC -> Bangkok  | ลงท้าย Z คือ UTC แปลงเป็นเวลากรุงเทพ
                             order_status=r["orderStatus"].upper(),
                             platform_item_name=it["itemName"], menu_item_nk=None,
                             quantity=int(it["quantity"]), unit_price_thb=float(it["itemPrice"]),
                             ingest_ts=meta.ingest_ts, source_file=meta.source_file))
    return pd.DataFrame(rows, columns=COMMON)

def adapt_lineman(raw: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for meta, r in zip(raw.itertuples(), raw.payload.map(json.loads)):
        for it in r["menu"]:
            rows.append(dict(source_system="LINEMAN", external_order_id=r["order_no"], channel_code="LINEMAN",
                             order_ts=pd.Timestamp(datetime.strptime(r["created_at"], "%d/%m/%Y %H:%M"), tz=BKK),
                             order_status="COMPLETED",                      # LINE MAN's export only contains completed orders  |  export ของ LINE MAN มีแต่ออร์เดอร์ที่สำเร็จ
                             platform_item_name=it["name"], menu_item_nk=None,
                             quantity=int(it["qty"]), unit_price_thb=float(it["subtotal"]) / int(it["qty"]),   # LINE MAN gives line subtotal, not unit price  |  LINE MAN ให้ยอดรวมต่อบรรทัด ไม่ใช่ราคาต่อชิ้น
                             ingest_ts=meta.ingest_ts, source_file=meta.source_file))
    return pd.DataFrame(rows, columns=COMMON)

ADAPTERS = {"pos": adapt_pos, "grab": adapt_grab, "lineman": adapt_lineman}

In [8]:
def transform(bday: str) -> dict:
    bday = pd.Timestamp(bday).date()

    # 1. read everything in Raw, run each source through its adapter  |  อ่าน Raw ทั้งหมด ผ่าน adapter ของแต่ละแหล่ง
    parts = []
    for source, adapter in ADAPTERS.items():
        files = list((LAKE / "raw" / source).rglob("*.parquet"))
        if files:
            parts.append(adapter(pd.concat(pd.read_parquet(f) for f in files)))
    lines = pd.concat(parts, ignore_index=True)

    # 2./3. one timezone (adapters did it), then business day; keep only the day we are loading
    # timezone เดียว (adapter ทำแล้ว) แล้วคำนวณวันทำการ เก็บเฉพาะวันที่กำลังโหลด
    lines["business_date"] = business_date(lines.order_ts)
    lines = lines[lines.business_date == bday].copy()
    n_in = len(lines)

    # 4. conform item names through the single map  |  แปลงชื่อเมนูผ่านตารางแมปตารางเดียว
    lines = lines.merge(item_name_map, on=["channel_code", "platform_item_name"], how="left", suffixes=("", "_map"))
    lines["menu_item_nk"] = lines.menu_item_nk.fillna(lines.menu_item_nk_map)
    lines = lines.drop(columns="menu_item_nk_map")

    # 5. dedupe on natural key: drop exact duplicate rows, then keep the latest snapshot of each order
    # ตัดซ้ำด้วย natural key: ทิ้งแถวที่ซ้ำเป๊ะ แล้วเก็บ snapshot ล่าสุดของแต่ละออร์เดอร์
    lines = lines.drop_duplicates(subset=["source_system", "external_order_id", "platform_item_name", "quantity", "unit_price_thb"])
    latest = lines.groupby(["source_system", "external_order_id"]).ingest_ts.transform("max")
    lines = lines[lines.ingest_ts == latest].copy()
    n_dedup = n_in - len(lines)

    # 6. quality gates: stop the job rather than publish bad numbers  |  ด่านคุณภาพ: หยุดงานดีกว่าปล่อยตัวเลขผิดออกไป
    unmapped = lines[lines.menu_item_nk.isna()]
    assert unmapped.empty, f"unmapped items, add them to item_name_map:\n{unmapped[['channel_code','platform_item_name']]}"
    assert lines.channel_code.notna().all(), "order without channel"
    assert (lines.quantity > 0).all() and (lines.unit_price_thb > 0).all(), "non-positive quantity or price"

    # 7. star schema  |  สร้าง fact และ dim
    lines["line_net_thb"] = lines.quantity * lines.unit_price_thb
    orders = (lines.groupby(["source_system", "external_order_id", "channel_code", "order_ts", "business_date", "order_status"], as_index=False)
                   .agg(gross_amount_thb=("line_net_thb", "sum")))
    orders["is_cancelled"] = orders.order_status.eq("CANCELLED")
    orders = orders.sort_values("order_ts").reset_index(drop=True)
    orders.insert(0, "order_sk", range(1, len(orders) + 1))

    lines = lines.merge(orders[["source_system", "external_order_id", "order_sk"]], on=["source_system", "external_order_id"])
    lines = lines.merge(menu_items[["menu_item_nk"]].assign(menu_item_sk=range(1, len(menu_items) + 1)), on="menu_item_nk")
    lines = lines[["order_sk", "menu_item_sk", "menu_item_nk", "quantity", "unit_price_thb", "line_net_thb"]].reset_index(drop=True)
    lines.insert(0, "order_line_sk", range(1, len(lines) + 1))

    # load (this notebook overwrites the day's partition; a real job would MERGE on order_sk)
    # โหลด (โน้ตบุ๊กนี้เขียนทับทั้งวัน งานจริงจะ MERGE ด้วย order_sk)
    con.execute("CREATE OR REPLACE TABLE trusted.dim_menu_item AS SELECT row_number() OVER () AS menu_item_sk, * FROM menu_items")
    con.execute("CREATE OR REPLACE TABLE trusted.dim_ingredient_recipe AS SELECT * FROM recipes")
    con.execute("CREATE OR REPLACE TABLE trusted.fct_order AS SELECT * FROM orders")
    con.execute("CREATE OR REPLACE TABLE trusted.fct_order_line AS SELECT * FROM lines")
    return dict(business_date=str(bday), lines_in=n_in, duplicates_removed=n_dedup, orders=len(orders), order_lines=len(lines))

transform("2026-09-11")

{'business_date': '2026-09-11',
 'lines_in': 46,
 'duplicates_removed': 18,
 'orders': 19,
 'order_lines': 28}

In [9]:
print("The after-midnight order P-1012 now belongs to Friday, and the Grab order is in Bangkok time:")
display(con.execute("""
    SELECT external_order_id, channel_code, order_ts, business_date, gross_amount_thb, is_cancelled
    FROM trusted.fct_order
    WHERE external_order_id IN ('P-1012', 'GF-88117', 'LM-5501')
""").df())

print("Three different names for the same product now share one key:")
display(con.execute("""
    SELECT o.channel_code, l.menu_item_nk, sum(l.quantity)::INT AS units
    FROM trusted.fct_order_line l JOIN trusted.fct_order o USING (order_sk)
    WHERE l.menu_item_nk = 'PEP-L' AND NOT o.is_cancelled
    GROUP BY 1, 2 ORDER BY 1
""").df())

The after-midnight order P-1012 now belongs to Friday, and the Grab order is in Bangkok time:


,external_order_id,channel_code,order_ts,business_date,gross_amount_thb,is_cancelled
0,LM-5501,LINEMAN,2026-09-11 12:20:00+07:00,2026-09-11,399.0,False
1,GF-88117,GRAB,2026-09-11 19:42:00+07:00,2026-09-11,359.0,False
2,P-1012,QR_DINEIN,2026-09-12 00:40:00+07:00,2026-09-11,329.0,False


Three different names for the same product now share one key:


,channel_code,menu_item_nk,units
0,GRAB,PEP-L,2
1,LINEMAN,PEP-L,1
2,PHONE,PEP-L,3
3,QR_DINEIN,PEP-L,2


## 5 · Refined zone

The 03:30 job. **Flows in:** Trusted facts and dims. **Processed:** aggregate to
business day × item × channel, and explode sales through recipes into ingredient usage.
**Stored:** one small table per question. Kept forever.

**Why it exists:** the Monday dashboard reads one small table instead of joining six large ones
on every click. And no business rule is applied here; cancellations were already flagged in Trusted,
so the mart just filters on the flag.

งานตอน 03:30 **ไหลเข้า:** fact และ dim จาก Trusted **ประมวลผล:** รวมยอดเป็น วันทำการ × เมนู × ช่องทาง
และกระจายยอดขายผ่านสูตรเป็นปริมาณวัตถุดิบที่ใช้ **เก็บเป็น:** หนึ่งตารางเล็ก ๆ ต่อหนึ่งคำถาม เก็บตลอดไป

**ทำไมต้องมี:** แดชบอร์ดวันจันทร์อ่านตารางเล็กตารางเดียว แทนที่จะ join ตารางใหญ่หกตารางทุกครั้งที่คลิก
และไม่มีกฎธุรกิจตรงนี้ ออร์เดอร์ที่ยกเลิกถูกติดธงไว้แล้วใน Trusted mart แค่กรองตามธง

In [10]:
def aggregate():
    con.execute("""
        CREATE OR REPLACE TABLE refined.mart_sales_daily AS
        SELECT o.business_date, o.channel_code, l.menu_item_nk, m.item_name, m.size_code,
               sum(l.quantity)::INT AS units_sold,
               sum(l.line_net_thb)  AS net_sales_thb,
               count(DISTINCT o.order_sk) AS orders
        FROM trusted.fct_order_line l
        JOIN trusted.fct_order o USING (order_sk)
        JOIN trusted.dim_menu_item m USING (menu_item_sk)
        WHERE NOT o.is_cancelled
        GROUP BY ALL ORDER BY 1, 2, 3
    """)
    con.execute("""
        CREATE OR REPLACE TABLE refined.mart_ingredient_usage_daily AS
        SELECT s.business_date, r.ingredient_nk,
               sum(s.units_sold * r.qty_per_unit)::INT AS theoretical_usage
        FROM refined.mart_sales_daily s
        JOIN trusted.dim_ingredient_recipe r USING (menu_item_nk)
        GROUP BY ALL ORDER BY 1, 3 DESC
    """)
    # also keep a Parquet copy in the zone folder, so the lake on disk is complete  |  เก็บสำเนา Parquet ไว้ในโฟลเดอร์ zone ด้วย
    for mart in ["mart_sales_daily", "mart_ingredient_usage_daily"]:
        con.execute(f"COPY refined.{mart} TO 'lake/refined/{mart}.parquet' (FORMAT PARQUET)")

aggregate()
tree(LAKE / "refined")
print()
display(con.execute("SELECT * FROM refined.mart_sales_daily").df())

mart_ingredient_usage_daily.parquet
mart_sales_daily.parquet



,business_date,channel_code,menu_item_nk,item_name,size_code,units_sold,net_sales_thb,orders
0,2026-09-11,GRAB,COKE,Coke 325ml,-,2,80.0,1
1,2026-09-11,GRAB,MAR-M,Margherita,M,1,239.0,1
2,2026-09-11,GRAB,PEP-L,Pepperoni,L,2,718.0,2
3,2026-09-11,GRAB,PEP-M,Pepperoni,M,2,570.0,1
4,2026-09-11,LINEMAN,COKE,Coke 325ml,-,1,40.0,1
5,2026-09-11,LINEMAN,HAW-L,Hawaiian,L,1,339.0,1
6,2026-09-11,LINEMAN,MAR-M,Margherita,M,2,478.0,1
7,2026-09-11,LINEMAN,PEP-L,Pepperoni,L,1,359.0,1
8,2026-09-11,LINEMAN,PEP-M,Pepperoni,M,1,285.0,1
9,2026-09-11,PHONE,HAW-L,Hawaiian,L,1,309.0,1


## 6 · Consumers

Nobody queries Trusted directly. The owner's dashboard and the reorder list read the marts.
This is the Monday-morning view for one day; the real report would cover seven.

ไม่มีใคร query Trusted ตรง ๆ แดชบอร์ดของเจ้าของร้านและรายการสั่งของอ่านจาก mart
นี่คือมุมมองเช้าวันจันทร์ของหนึ่งวัน รายงานจริงจะครอบคลุมเจ็ดวัน

In [11]:
print("Sales by channel (what the owner looks at on Monday):")
display(con.execute("""
    SELECT channel_code, sum(orders)::INT AS orders, sum(units_sold)::INT AS units, sum(net_sales_thb) AS net_sales_thb
    FROM refined.mart_sales_daily GROUP BY 1 ORDER BY 4 DESC
""").df())

print("Ingredient usage for the day (feeds the nightly reorder list):")
display(con.execute("""
    SELECT ingredient_nk, theoretical_usage,
           CASE WHEN ingredient_nk = 'coke_can' THEN 'cans' ELSE 'grams' END AS unit
    FROM refined.mart_ingredient_usage_daily
""").df())

Sales by channel (what the owner looks at on Monday):


,channel_code,orders,units,net_sales_thb
0,QR_DINEIN,14,19,2975.0
1,GRAB,5,7,1607.0
2,LINEMAN,5,6,1501.0
3,PHONE,3,4,1296.0


Ingredient usage for the day (feeds the nightly reorder list):


,ingredient_nk,theoretical_usage,unit
0,flour_00,5940,grams
1,mozzarella,3080,grams
2,tomato_sauce,1900,grams
3,pepperoni,795,grams
4,pineapple,240,grams
5,ham,180,grams
6,basil,18,grams
7,coke_can,12,cans


## 7 · Why we keep Raw: the replay

A month later we find a bug in the transform. Fixing it does not mean asking Grab for last
month's files again. We drop Trusted and Refined, run the same two jobs from Raw, and get the
same numbers. Landing could have been emptied long ago; Raw is enough.

หนึ่งเดือนต่อมาเราเจอบั๊กใน transform การแก้ไม่ได้แปลว่าต้องไปขอไฟล์เดือนที่แล้วจาก Grab ใหม่
เราลบ Trusted และ Refined ทิ้ง รันสองงานเดิมจาก Raw แล้วได้ตัวเลขเท่าเดิม Landing จะถูกล้างไปนานแล้วก็ไม่เป็นไร แค่ Raw ก็พอ

In [12]:
before = con.execute("SELECT sum(net_sales_thb) FROM refined.mart_sales_daily").fetchone()[0]

con.execute("DROP TABLE trusted.fct_order"); con.execute("DROP TABLE trusted.fct_order_line")
con.execute("DROP TABLE refined.mart_sales_daily"); con.execute("DROP TABLE refined.mart_ingredient_usage_daily")
shutil.rmtree(LAKE / "landing"); (LAKE / "landing").mkdir()     # landing is gone too, as it would be after 7 days  |  Landing หายไปแล้วด้วย เหมือนหลังผ่านไป 7 วัน

stats = transform("2026-09-11")
aggregate()
after = con.execute("SELECT sum(net_sales_thb) FROM refined.mart_sales_daily").fetchone()[0]

print(stats)
print(f"net sales before replay: {before:,.0f} THB, after replay from Raw: {after:,.0f} THB")
assert before == after

{'business_date': '2026-09-11', 'lines_in': 46, 'duplicates_removed': 18, 'orders': 19, 'order_lines': 28}
net sales before replay: 7,379 THB, after replay from Raw: 7,379 THB


## 8 · One order, traced

The same Grab order as in the HTML page: a large pepperoni on Friday 19:42, order `GF-88117`.
Here is where that single record sits in each zone.

ออร์เดอร์ Grab เดียวกับในหน้า HTML: เปปเปอโรนีถาดใหญ่ วันศุกร์ 19:42 ออร์เดอร์ `GF-88117` ดูว่าเรคคอร์ดเดียวนี้อยู่ตรงไหนในแต่ละ zone

In [13]:
oid = "GF-88117"
print("RAW      :", con.execute(f"""SELECT source_file, json_extract_string(payload, '$.orderTime') AS utc,
                                           json_extract_string(payload, '$.items[0].itemName') AS item, count(*) AS copies
                                    FROM read_parquet('lake/raw/grab/*/*.parquet')
                                    WHERE json_extract_string(payload, '$.orderID') = '{oid}' GROUP BY ALL""").fetchone())
print("TRUSTED  :", con.execute(f"""SELECT o.external_order_id, o.order_ts, o.business_date, l.menu_item_nk, l.quantity
                                    FROM trusted.fct_order o JOIN trusted.fct_order_line l USING (order_sk)
                                    WHERE o.external_order_id = '{oid}'""").fetchone())
print("REFINED  :", con.execute("""SELECT business_date, channel_code, menu_item_nk, units_sold
                                   FROM refined.mart_sales_daily WHERE channel_code = 'GRAB' AND menu_item_nk = 'PEP-L'""").fetchone())
print("USAGE    :", con.execute("""SELECT ingredient_nk, theoretical_usage FROM refined.mart_ingredient_usage_daily
                                   WHERE ingredient_nk = 'mozzarella'""").fetchone(), "<- includes this pizza's 150 g")

RAW      : ('orders_2026-09-11.json', '2026-09-11T12:42:00Z', 'Large Pepperoni Pizza', 2)
TRUSTED  : ('GF-88117', datetime.datetime(2026, 9, 11, 19, 42, tzinfo=<DstTzInfo 'Asia/Bangkok' +07+7:00:00 STD>), datetime.date(2026, 9, 11), 'PEP-L', 1)
REFINED  : (datetime.date(2026, 9, 11), 'GRAB', 'PEP-L', 2)
USAGE    : ('mozzarella', 3080) <- includes this pizza's 150 g


## 9 · What this notebook is not

- **Not a scheduler.** In production, a cron or Airflow DAG calls `land`, `ingest`, `transform`
  and `aggregate` on the cadences in the HTML page (POS every 5 min, exports at 06:00,
  transform 03:00, aggregate 03:30). The functions would be the same.
- **Not incremental.** `transform` rebuilds one business day. A real job would `MERGE` into the
  facts instead of `CREATE OR REPLACE`, and `dim_menu_item` would be SCD2.
- **Not the full model.** Payments, modifiers, promotions, stock counts, shifts and the other
  three marts follow exactly the same path; they are left out to keep the flow readable.
- **No sandbox.** The lecture's zone pattern has one for analyst experiments. With one analyst and
  DuckDB, a local copy of a mart is the sandbox, so the notebook does not build a zone for it.

The one-line summary still holds: **collect often, keep everything raw, clean once, aggregate once,
read many times.**

โน้ตบุ๊กนี้ **ไม่ใช่**:
- **ตัวตั้งเวลา** ในระบบจริง cron หรือ Airflow จะเรียก `land`, `ingest`, `transform`, `aggregate` ตามความถี่ในหน้า HTML (POS ทุก 5 นาที, export ตอน 06:00, transform 03:00, aggregate 03:30) ตัวฟังก์ชันเหมือนเดิม
- **incremental** `transform` สร้างใหม่ทั้งวันทำการ งานจริงจะ `MERGE` เข้า fact แทน `CREATE OR REPLACE` และ `dim_menu_item` จะเป็น SCD2
- **โมเดลเต็ม** การชำระเงิน modifier โปรโมชัน นับสต็อก กะพนักงาน และ mart อีกสามตัว เดินเส้นทางเดียวกันเป๊ะ ตัดออกเพื่อให้อ่านง่าย
- **ไม่มี sandbox** แพตเทิร์น zone ในบทเรียนมี sandbox ไว้ให้นักวิเคราะห์ทดลอง แต่เรามีนักวิเคราะห์คนเดียวและใช้ DuckDB สำเนา mart ในเครื่องก็คือ sandbox แล้ว โน้ตบุ๊กจึงไม่สร้าง zone แยก

สรุปบรรทัดเดียวยังใช้ได้: **เก็บบ่อย เก็บดิบไว้ทั้งหมด ทำความสะอาดครั้งเดียว รวมยอดครั้งเดียว อ่านได้หลายครั้ง**

In [14]:
con.close()
print("Final lake on disk:\n")
tree(LAKE)

Final lake on disk:

landing/
raw/
    grab/
        ingest_date=2026-09-12/
            orders_2026-09-11.parquet
    lineman/
        ingest_date=2026-09-12/
            export_20260912.parquet
    pos/
        ingest_date=2026-09-11/
            orders_20260911T1110.parquet
            orders_20260911T1115.parquet
            orders_20260911T1205.parquet
            orders_20260911T1245.parquet
            orders_20260911T1250.parquet
            orders_20260911T1830.parquet
            orders_20260911T1900.parquet
            orders_20260911T1905.parquet
            orders_20260911T1940.parquet
            orders_20260911T2015.parquet
            orders_20260911T2100.parquet
            orders_20260911T2105.parquet
            orders_20260911T2250.parquet
        ingest_date=2026-09-12/
            orders_20260912T0040.parquet
refined/
    mart_ingredient_usage_daily.parquet
    mart_sales_daily.parquet
trusted/
    warehouse.duckdb
